# Resident Risk Explanatory

## Problem Framing

**Business question:** Which care and safety signals most explain elevated resident risk?

**Operational decision supported:** Trigger earlier staff intervention for residents with elevated short-term safety risk.

**Primary users:** case managers, operations staff

**Target or analysis anchor:** Current predictive label: `label_incident_next_30d` from resident-monthly snapshots.

This standardized explanatory notebook keeps the narrative focused on decisions and evidence while reusing the same shared platform as the predictive work.


## Shared Assets And Notebook Bootstrap

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-2-shared-prep.md`
* `ml/docs/phase-b-notebook-standardization.md`


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [2]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='resident_risk',
    dataset_name='resident_monthly_features',
    predictive_impl='resident_risk',
    use_predictive_dataset=False,
)
dataset = context["dataset"]

summarize_frame(dataset)


,resident_id,snapshot_month,safehouse_id,case_status,case_category,sex,initial_risk_level,months_since_admission,age_years_at_snapshot,subcategory_flag_count,...,future_window_complete_60d,future_window_complete_90d,future_window_complete_120d,label_incident_next_30d,label_case_prioritization_next_60d,label_counseling_progress_next_90d,label_education_improvement_next_120d,label_wellbeing_deterioration_next_90d,label_supportive_home_visit_next_120d,label_reintegration_complete_next_90d
0,1,2023-10-01,4,Active,Neglected,F,Critical,0,15.08,0,...,True,True,True,False,False,True,True,False,False,False
1,1,2023-11-01,4,Active,Neglected,F,Critical,1,15.17,0,...,True,True,True,False,True,True,True,False,False,False
2,1,2023-12-01,4,Active,Neglected,F,Critical,2,15.25,0,...,True,True,True,False,True,True,False,False,False,False
3,1,2024-01-01,4,Active,Neglected,F,Critical,3,15.33,0,...,True,True,True,False,True,True,False,False,False,False
4,1,2024-02-01,4,Active,Neglected,F,Critical,4,15.42,0,...,True,True,True,True,True,True,True,False,False,False
5,1,2024-03-01,4,Active,Neglected,F,Critical,5,15.50,0,...,True,True,True,False,True,True,False,False,False,False
6,1,2024-04-01,4,Active,Neglected,F,Critical,6,15.58,0,...,True,True,True,False,True,False,False,False,False,False
7,1,2024-05-01,4,Active,Neglected,F,Critical,7,15.67,0,...,True,True,True,False,True,False,False,False,False,False
8,1,2024-06-01,4,Active,Neglected,F,Critical,8,15.75,0,...,True,True,True,True,True,True,False,False,False,False
9,1,2024-07-01,4,Active,Neglected,F,Critical,9,15.83,0,...,True,True,True,False,True,True,False,False,False,False


## Standard Analysis Plan

1. Review the shared table grain and confirm it matches the business question.
2. Compare segments that matter operationally, not just statistically.
3. Reuse the nearest predictive artifact when it helps explain feature importance or threshold behavior.
4. Close with recommendations the application can actually surface.


## Evidence Review

Use the nearest predictive artifact only when it genuinely clarifies the analysis. Keep the explanatory notebook focused on evidence the user can act on, not on repeating a model leaderboard.


In [3]:
evaluation = load_evaluation_bundle('resident_risk')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


{'best_model_name': 'logistic_regression',
 'train_rows': 756,
 'test_rows': 777,
 'target_col': 'label_incident_next_30d',
 'split_col': 'snapshot_month',
 'selection_metric': 'average_precision',
 'cutoff_date': '2025-04-01',
 'task_type': 'classification',
 'sample_count': 777.0,
 'positive_count': 36.0,
 'positive_rate': 0.04633204633204633,
 'accuracy': 0.8584298584298584,
 'balanced_accuracy': 0.5822087269455691,
 'precision': 0.10638297872340426,
 'recall': 0.2777777777777778,
 'f1': 0.15384615384615385,
 'roc_auc': 0.7580971659919028,
 'average_precision': 0.1231033816134102}

In [4]:
summarize_frame(holdout_comparison)


,pipeline_name,model_name,sample_count,positive_count,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,mae,rmse,r2
0,resident_risk,logistic_regression,777.0,36.0,0.046332,0.886744,0.663124,0.182927,0.416667,0.254237,0.903959,0.218096,NaN,NaN,NaN
1,resident_risk,random_forest_classifier,777.0,36.0,0.046332,0.953668,0.500000,0.000000,0.000000,0.000000,0.855357,0.186471,NaN,NaN,NaN
2,resident_risk,dummy_classifier,777.0,36.0,0.046332,0.953668,0.500000,0.000000,0.000000,0.000000,0.500000,0.046332,NaN,NaN,NaN


## Business Interpretation

Explain:

1. Which patterns matter most for the decision this notebook supports.
2. Why those patterns matter for case managers, operations staff.
3. Which actions should change in planning, outreach, or case management.
4. Which limitations mean the analysis should stay explanation-first for now.


## Deployment Notes

Recommended shared widgets:

* `risk_badge_widget`
* `ranked_table_widget`
* `explanation_chart_card`

Deployment checklist:

* Expose risk badges in resident detail views and triage dashboards.
* Use the same explanation card in watchlists and alert workflows.

Preferred delivery pattern:

* use explanation chart cards and insight summaries first
* only add new endpoints if the analysis becomes user-facing and interactive
